# 23 - Analogo en R (recreado desde cero)

## Tema base

**23 - K-Means: clustering, elbow y silhouette con flujo completo**


## Objetivos de la sesion en R

1. Mantener teoria del notebook Python en equivalente R.
2. Usar codigo 100% R ejecutable.
3. Conservar ejercicios adaptados a R.
4. Integrar extra de probabilidad y estadistica.


## Ruta Teoria-Codigo (alternada)

Se organiza la sesion en bloques cortos que alternan concepto y practica.


### Bloque teorico 1

### Teoria 1

# 23 - K-Means: clustering, elbow y silhouette con flujo completo

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que resuelve K-Means dentro de ciencia de datos.
2. Conocer el algoritmo paso a paso.
3. Interpretar formulas clave como inercia.
4. Aplicar un flujo con `train_test_split` en clustering.
5. Usar e interpretar las metricas de elbow y silhouette.
6. Reconocer riesgos y malas practicas comunes.

### Teoria 2

## 1) Recordatorio rapido: ciencia de datos y clustering

Ciencia de datos busca apoyar decisiones con evidencia cuantitativa.

En problemas **supervisados** tienes una variable objetivo (ej. clase o precio).
En problemas **no supervisados** no hay etiqueta; quieres encontrar estructura oculta.

K-Means es un metodo no supervisado para agrupar observaciones parecidas.

### Teoria 3

## 2) Que es K-Means

K-Means intenta dividir los datos en `k` grupos.
Cada grupo se representa por su centroide (promedio de puntos del grupo).

Idea intuitiva:

1. Propone `k` centroides iniciales.
2. Asigna cada punto al centroide mas cercano.
3. Recalcula centroides como promedio de puntos asignados.
4. Repite 2-3 hasta converger.


In [ ]:
if(!requireNamespace('cluster', quietly=TRUE)) install.packages('cluster', repos='https://cloud.r-project.org')
library(cluster)
set.seed(42)
n <- 900
centros <- matrix(c(4,120, 8,260, 14,420, 10,620), byrow=TRUE, ncol=2)
z <- sample(1:4, n, replace=TRUE)
X <- cbind(frecuencia_mensual=rnorm(n, centros[z,1], 1.2), ticket_promedio=rnorm(n, centros[z,2], 45))


### Bloque teorico 2

### Teoria 4

## 3) Formulas clave

### 3.1 Funcion objetivo de K-Means

$$\min_{C_1,\dots,C_k,\mu_1,\dots,\mu_k} \sum_{r=1}^{k} \sum_{x_i \in C_r} ||x_i - \mu_r||_2^2$$

donde:

- $C_r$ es el conjunto de puntos en el cluster $r$.
- $\mu_r$ es el centroide del cluster $r$.

### 3.2 Actualizacion del centroide

$$\mu_r = \frac{1}{|C_r|} \sum_{x_i \in C_r} x_i$$

### 3.3 Inercia (WCSS/SSE)

En `scikit-learn`, `inertia_` representa:

$$\text{Inercia} = \sum_{i=1}^{n} \min_{r \in \{1,\dots,k\}} ||x_i - \mu_r||_2^2$$

Menor inercia implica clusters mas compactos respecto a sus centroides.

### Teoria 5

## 4) Aplicaciones interesantes de K-Means

1. Segmentacion de clientes (marketing y ofertas).
2. Agrupacion de zonas geograficas por comportamiento.
3. Agrupar documentos por temas aproximados.
4. Compresion de imagen por cuantizacion de colores.
5. Deteccion preliminar de patrones atipicos (clusters pequenos o lejanos).

Hoy simularemos un caso de segmentacion de clientes con 2 variables de negocio.

### Teoria 6

## 5) Datos de ejemplo: segmentacion sintetica

Creamos dos variables con significado de negocio:

- `frecuencia_mensual`: numero de compras al mes.
- `ticket_promedio_usd`: gasto promedio por compra.

Las escalas son distintas a proposito para mostrar por que normalizar importa.


In [ ]:
set.seed(2026)
idx <- sample(seq_len(nrow(X)), floor(0.75*nrow(X)))
X_train <- X[idx,]; X_test <- X[-idx,]
mu <- colMeans(X_train); sd_ <- apply(X_train,2,sd)
scale_with <- function(M, mu, sd_) sweep(sweep(as.matrix(M),2,mu,'-'),2,sd_,'/')
unscale_with <- function(Ms, mu, sd_) sweep(sweep(as.matrix(Ms),2,sd_,'*'),2,mu,'+')
X_train_sc <- scale_with(X_train,mu,sd_)


### Bloque teorico 3

### Teoria 7

## 6) Flujo con train/test split en clustering

En clustering no siempre se hace train/test como en clasificacion.
Aun asi, puede ser util para:

1. elegir hiperparametros sin usar todo el dataset,
2. evaluar estabilidad de estructura en datos no vistos,
3. evitar fuga de informacion al escalar (fit scaler solo en train).

### Teoria 8

## 7) Impacto de normalizar: comparacion rapida

Comparamos `k=4` con y sin normalizar.
Usamos silhouette para comparar separacion/cohesion bajo cada geometria.

### Teoria 9

## 8) Elbow y Silhouette para elegir `k`

### 8.1 Elbow (codo)

Elbow observa la inercia al variar `k`.
Buscas un punto donde agregar mas clusters ya mejora poco.

### 8.2 Silhouette

Para cada punto `i`:

- $a(i)$ = distancia promedio a puntos de su propio cluster.
- $b(i)$ = menor distancia promedio a otro cluster (el mas cercano alternativo).

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$

Rango de silhouette: `[-1, 1]`.

- cercano a 1: bien ubicado,
- cerca de 0: en frontera,
- negativo: posible mala asignacion.


In [ ]:
k_values <- 2:10
inertia <- sil <- numeric(length(k_values))
for(i in seq_along(k_values)){k <- k_values[i]; km <- kmeans(X_train_sc, centers=k, nstart=30); inertia[i] <- km$tot.withinss; sil[i] <- mean(silhouette(km$cluster, dist(X_train_sc))[, 'sil_width'])}
cbind(k=k_values, inertia=round(inertia,1), silhouette=round(sil,3))


### Bloque teorico 4

### Teoria 10

## 9) Por que elbow baja cuando `k` aumenta

La inercia es una suma de distancias cuadradas al centroide mas cercano.
Al aumentar `k`, das mas centroides disponibles para aproximar los datos.

Formalmente, el problema con `k+1` clusters contiene al de `k` como caso particular.
Por eso el minimo alcanzable de inercia con `k+1` **no puede ser mayor** que con `k`.

Conclusion: la curva elbow es monotona no creciente.
Lo dificil no es que baje, sino decidir en que punto deja de bajar "lo suficiente".

### Teoria 11

## 10) Puntos clave y puntos de cuidado

### Puntos clave

1. Normalizar variables suele ser obligatorio en K-Means.
2. Elbow y silhouette son guias, no reglas absolutas.
3. `random_state` y `n_init` importan para estabilidad.

### Puntos de cuidado

1. K-Means asume clusters aproximadamente esfericos y de tamano comparable.
2. Es sensible a outliers (mueven centroides).
3. Si hay estructura no convexa, puede fallar (ej. formas de media luna).
4. Menor inercia no siempre implica mejor interpretacion de negocio.
5. Elegir `k` solo por grafica sin contexto puede llevar a decisiones malas.

### Teoria 12

## 12) Cierre

Ideas clave:

1. K-Means minimiza distancias cuadraticas a centroides.
2. La inercia siempre baja al subir `k`, por construccion del problema.
3. Silhouette complementa elbow al medir separacion/cohesion relativa.
4. Normalizar y controlar inicializacion son practicas criticas.
5. Elegir `k` requiere evidencia tecnica + criterio de negocio.


In [ ]:
par(mfrow=c(1,2))
plot(k_values, inertia, type='b', pch=16, col='steelblue', xlab='k', ylab='inercia', main='Elbow')
plot(k_values, sil, type='b', pch=16, col='darkorange', xlab='k', ylab='silhouette', main='Silhouette')
par(mfrow=c(1,1))


## Extra de probabilidad y estadistica

Inercia, elbow, silhouette y estabilidad de particiones.


In [ ]:
k_best <- k_values[which.max(sil)]
km_final <- kmeans(X_train_sc, centers=k_best, nstart=50)
centers_orig <- unscale_with(km_final$centers, mu, sd_)
plot(X_train[,1], X_train[,2], col=km_final$cluster, pch=19, xlab='frecuencia_mensual', ylab='ticket_promedio')
points(centers_orig[,1], centers_orig[,2], pch=8, cex=2.2, lwd=3, col='black')


## Analogias R y Python

- `kmeans` en R optimiza la misma funcion objetivo que K-Means en Python.
- Elbow/silhouette son heuristicas en ambos.


In [ ]:
# Checklist rapido R vs Python
# 1) ?Que cambia en sintaxis?
# 2) ?Que cambia en estructura de datos?
# 3) ?Que cambia en manejo de NA/errores?


## Errores comunes

- Elegir k solo por estetica del codo.
- No normalizar.
- Ignorar outliers y forma real de clusters.


In [ ]:
# Auto-verificacion de errores comunes
# Define un caso borde y valida que tu solucion en R no falle silenciosamente.


## Ejercicios para pensar (no copiar codigo)

Primero argumenta en texto. Luego, si hace falta, valida con experimentos en R.


### Ejercicio reflexivo 1

### Ejercicio 1

## Ruta de la sesion

1. Recordatorio breve de ciencia de datos.
2. Que es K-Means y cuando usarlo.
3. Formulas clave (objetivo, centroides, inercia).
4. Por que la curva elbow baja cuando aumenta `k`.
5. Que mide silhouette.
6. Flujo de extremo a extremo con datos de segmentacion sintetica.
7. Visualizaciones de apoyo.
8. Puntos clave y puntos de cuidado.
9. Ejercicios para pensar.


### Ejercicio reflexivo 2

### Ejercicio 2

## 11) Ejercicios de pensamiento (no copiar/pegar)

Primero escribe tu argumento en texto. Luego valida con codigo.


### Ejercicio reflexivo 3

### Ejercicio 3

### Ejercicio 1: elbow vs silhouette en conflicto

Supongamos que elbow sugiere `k=4`, pero silhouette maxima ocurre en `k=3`.

1. Que decision tomarias y por que.
2. Que informacion de negocio pedirias para decidir.
3. Que experimento adicional correria tu equipo.


### Ejercicio reflexivo 4

### Ejercicio 4

### Ejercicio 2: sensibilidad a outliers

Agrega manualmente 8 puntos extremos al dataset y repite el analisis.

1. Como cambian centroides.
2. Como cambian elbow y silhouette.
3. Que estrategia propones para robustecer el flujo.


### Ejercicio reflexivo 5

### Ejercicio 5

### Ejercicio 3: impacto de no normalizar

Disena una prueba para mostrar como domina la variable de mayor escala.

1. Multiplica `ticket_promedio_usd` por 10.
2. Repite clustering sin escalar y con escalar.
3. Explica por que cambian (o no) los clusters.


### Ejercicio reflexivo 6

### Ejercicio 6

### Ejercicio 4: estabilidad de K-Means

Cambia `random_state` y `n_init`.

1. Cuanto se mueven los centroides entre corridas.
2. En que rango varia silhouette.
3. Que configuracion minima de `n_init` recomendarias y por que.


### Ejercicio reflexivo 7

### Ejercicio 7

### Ejercicio 5: traduccion a decision de negocio

Redacta una recomendacion para un equipo comercial:

1. cuantos segmentos usar,
2. como describir cada segmento en lenguaje no tecnico,
3. que accion concreta haria el equipo con cada segmento,
4. que riesgo existe si la distribucion de clientes cambia en 6 meses.


### Preguntas de cierre

1. ?Que supuesto estadistico es mas fragil en esta clase?
2. ?Que evidencia minima pedirias antes de usar este enfoque en un caso real?
3. ?Como explicarias este tema a alguien no tecnico sin perder rigor?
4. ?Que cambiaria entre una implementacion en R y una en Python para produccion?
